# Create a mock galaxy to be fit with Bagpipes and Prospector

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np 
import matplotlib as mpl
mpl.rcParams["font.family"] = "serif"  # override bagpipes' Helvetica request
mpl.rcParams["text.usetex"] = True

from astropy.cosmology import WMAP9 as cosmo
import bagpipes as pipes
import matplotlib.pyplot as plt
#%matplotlib inline

from astropy.io import fits
from astropy.table import Table
from sedpy import observate

import mockutils as mock

import json

In case of bugs with matplotlib

In [ ]:
import matplotlib as mpl
import os
import shutil

os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"
# This will find and delete the font cache folder
cache_dir = mpl.get_cachedir()
if os.path.exists(cache_dir):
#    shutil.rmtree(cache_dir)
    print("Cache cleared! Restart your Python kernel/IDE.")

latex_path = shutil.which("latex")
pdflatex_path = shutil.which("pdflatex")

print(f"LaTeX path: {latex_path}")
print(f"PDFLaTeX path: {pdflatex_path}")

if not latex_path:
    print("❌ LaTeX was not found in your system's PATH.")
else:
    print("✅ LaTeX is installed!")

# Generate Filter Transmission Curves

## HST

In [ ]:
print(observate.list_available_filters())

HST = ['acs_wfc_f435w', 'acs_wfc_f475w', 'acs_wfc_f555w', 'acs_wfc_f606w', 'acs_wfc_f625w', 'acs_wfc_f775w', 'acs_wfc_f814w', 'acs_wfc_f850lp', 'wfc3_ir_f098m', 'wfc3_ir_f105w', 'wfc3_ir_f110w', 'wfc3_ir_f125w', 'wfc3_ir_f140w', 'wfc3_ir_f160w']

hst_filter_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/filters/hst'
os.makedirs(hst_filter_dir, exist_ok=True)

data = observate.load_filters(HST)
for filt in data:
    name = filt.name
    wave = filt.wavelength
    trans = filt.transmission
    
    # Combine wavelength and transmission into a 2D array (column-wise)
    output_data = np.column_stack((wave, trans))
    
    # Define a filename based on the filter name
    
    filename = os.path.join(hst_filter_dir, name) + ".par"
    
    # Save to file with a helpful header
    if not os.path.exists(filename):
        np.savetxt(
            filename, 
            output_data, 
            #fmt=['%.4f', '%.6f'], 
            comments=''
        )
    
        print(f"Saved {filename}")

## JWST (NIRCam and MIRI)

Now automatically goes through all available filter transmission curves stored in sedpy and separates NIRCam from MIRI data points!

In [ ]:
import re

all_jwst_filters = [f for f in observate.list_available_filters() if 'jwst' in f]

nircam_filter_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/filters/nircam'
os.makedirs(nircam_filter_dir, exist_ok=True)

miri_filter_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/filters/miri'
os.makedirs(miri_filter_dir, exist_ok=True)

NIRCAM = []
MIRI = []

for jwst in all_jwst_filters:
    if 'niriss' in jwst or 'mod' in jwst:
        continue
    
    match = re.search(r'f(\d+)', jwst)
    if match:
        wavelength = int(match.group(1))
    else:
        print("No match found!")
        wavelength = 0

    if wavelength < 500:
        NIRCAM.append(jwst)
    else:
        MIRI.append(jwst)
            
print(f"NIRCam count: {len(NIRCAM)}")
print(f"MIRI count: {len(MIRI)}")

# For NIRCam
data = observate.load_filters(NIRCAM)
for filt in data:
    name = filt.name
    wave = filt.wavelength
    trans = filt.transmission
    
    # Combine wavelength and transmission into a 2D array (column-wise)
    output_data = np.column_stack((wave, trans))
    
    # Define a filename based on the filter name
    
    filename = os.path.join(nircam_filter_dir, name) + ".par"
    
    # Save to file with a helpful header
    np.savetxt(
        filename, 
        output_data, 
        #fmt=['%.4f', '%.6f'], 
        comments=''
    )
    
    print(f"Saved {filename}")
    
# For MIRI
data = observate.load_filters(MIRI)
for filt in data:
    name = filt.name
    wave = filt.wavelength
    trans = filt.transmission
    
    # Combine wavelength and transmission into a 2D array (column-wise)
    output_data = np.column_stack((wave, trans))
    
    # Define a filename based on the filter name
    
    filename = os.path.join(miri_filter_dir, name) + ".par"
    
    # Save to file with a helpful header
    np.savetxt(
        filename, 
        output_data, 
        #fmt=['%.4f', '%.6f'], 
        comments=''
    )
    
    print(f"Saved {filename}")

## ALMA bands 6 and 7 (boxcar)

In [ ]:
alma_filter_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/filters/alma'
os.makedirs(alma_filter_dir, exist_ok=True)

# Band 6
mock.write_alma_transmission_curves(file_path=os.path.join(alma_filter_dir, 'alma_band6'), 
                               central_freq_ghz=233, 
                               bandwidth_ghz=7.5)

# Band 7
mock.write_alma_transmission_curves(file_path=os.path.join(alma_filter_dir, 'alma_band7'), 
                               central_freq_ghz=343.5, 
                               bandwidth_ghz=7.5)

## Create a CSV file to organise my available filters

In [ ]:
import os

# Set your root filters directory
filter_base_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/filters'

# Initialize a master list to hold all paths
all_paths = []
# Dictionary to hold paths by instrument (e.g., {'nircam': [...], 'miri': [...]})
instrument_lists = {}

# Walk through the directories
for root, dirs, files in os.walk(filter_base_dir):    
    
    for f in files:
        # Get the instrument name from the folder name
        instrument = os.path.basename(root)
        
        # Create the relative path Bagpipes expects: "instrument/filename.dat"
        # Note: We use the instrument folder name as the prefix
        relative_path = os.path.join('filters', instrument, f)
        full_path = os.path.join(root, relative_path)
        
        print(relative_path)
        
        # Store in instrument-specific list
        if instrument not in instrument_lists:
            instrument_lists[instrument] = []
        instrument_lists[instrument].append(relative_path)
        
        # Add to master list
        all_paths.append(relative_path)

all_paths.sort()

# Write the "All" file
all_file = os.path.join(filter_base_dir, "all_filters.txt")
if not os.path.exists(all_file):
    with open(all_file, "w") as f:
        f.write('\n'.join(all_paths))

# Write individual instrument files
for instr, paths in instrument_lists.items():
    paths.sort()
    instr_file = os.path.join(filter_base_dir, f"{instr}_filters.txt")
    with open(instr_file, "w") as f:
        f.write('\n'.join(paths))

print(f"Generated filter lists for: {', '.join(instrument_lists.keys())} and 'all_filters.txt'")

# Set galaxy parameters

In [ ]:
mock_id = 9995

fit_instructions = {}                       # The fit instructions dictionary

# Nebular component
nebular = {}
nebular["logU"] = -3.0                # Log_10 of the ionisation parameter.
"""
exp = {}                          # Tau model star formation history component
exp["age"] = 2.2                 # Gyr
exp["tau"] = 0.75                 # Gyr
exp["massformed"] = 10.5            # log_10(M*/M_solar)
exp["metallicity"] = 0.1          # Z/Z_oldsolar
"""
lognormal = {}
lognormal["massformed"] = 11.0            # log_10(M*/M_solar)
lognormal["metallicity"] = 0.4          # Z/Z_oldsolar
lognormal["tmax"] = 2.0
lognormal["fwhm"] = 3.0

"""
dblplaw = {}
dblplaw["massformed"] = 11.0
dblplaw["metallicity"] = 0.4
dblplaw["tau"] = 2.5
dblplaw["alpha"] = 18.0
dblplaw["beta"] = 2.5
"""

zred = 2.50 # Set custom redshift

# Dust absorption parameters
dust = {}                               # Dust component
dust["type"] = "CF00"                   # Define the shape of the attenuation curve
dust["Av"] = 3.0                  # Vary Av between 0 and 3.2 magnitudes
dust["n"] = 0.5                 # Vary the slope of the attenuation curve from -1.0 to 1.5

# Dust emission parameters (now free parameters)
dust["qpah"] = 1.5                  # PAH mass fraction (spanning the entire grid)
dust["umin"] = 7.5                   # Lower limit of starlight intensity distribution (spanning the entire grid)
dust["gamma"] = 0.2                  # Fraction of stars at Umin

model_components = {}                   # The model components dictionary
model_components["nebular"] = nebular
model_components["redshift"] = zred      # Observed redshift  
#model_components["exponential"] = exp   
model_components["lognormal"] = lognormal
#model_components["dblplaw"] = dblplaw
model_components["dust"] = dust

print(model_components)

# Create model using all available bands for HST, JWST plus ALMA band 6&7
filt_list = np.loadtxt("filters/all_filters.txt", dtype="str")

model = pipes.model_galaxy(model_components, filt_list=filt_list, phot_units="mujy")

mock_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/data/mocks'

#Save to a file
with open(f"{mock_dir}/{mock_id}_model_components.json", "x") as f:
    json.dump(model_components, f, indent=4)

fig1, _ = model.plot()
fig1.savefig(f"{mock_dir}/{mock_id}_sed.png", bbox_inches='tight', dpi=300)

fig2, _ = model.sfh.plot()
fig2.savefig(f"{mock_dir}/{mock_id}_sfh.png", bbox_inches='tight', dpi=300)

plt.close('all') # Important: clear memory after saving

#fig = model.plot()
#fig = model.sfh.plot()
#model.plot_full_spectrum()

# Get model photometry

In [ ]:
import pandas as pd

true_flux = model.photometry # This is the 'noiseless' truth

flux_err = true_flux * 0.1  # Add 10% flux error and perturb the observations
mock_flux = np.random.normal(loc=true_flux, scale=flux_err)

# Quick diagnostic printout
for i in range(len(true_flux)):
    print(f"Filter {i:02d} | True: {true_flux[i]:.2e} | Mock Observed: {mock_flux[i]:.2e} +/- {flux_err[i]:.2e}") 

# Build the table
df = pd.DataFrame({
    'filter_name': [os.path.basename(f) for f in filt_list],
    'true_flux': true_flux,
    'mock_flux': mock_flux,
    'flux_err': flux_err
})

# Save it
df.to_csv(f"data/mocks/{mock_id}_mock_phot.csv", index=False, mode="x")
print("Mock photometry successfully saved.")